In [2]:
import os
os.chdir('D:/Projects/tumor-immune-scrna-atlas')
print(os.getcwd())

import scanpy as sc
import matplotlib.pyplot as plt
 
adata = sc.read_h5ad('data/interim/05b_integrated.h5ad')
 
# Compute UMAP for uncorrected PCA for the comparison panel
sc.pp.neighbors(adata, use_rep='X_pca', key_added='uncorr')
sc.tl.umap(adata, neighbors_key='uncorr')
adata.obsm['X_umap_uncorrected'] = adata.obsm['X_umap'].copy()
 
# Three-method × two-coloring grid
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
 
methods = [
    ('X_umap_uncorrected', 'Uncorrected'),
    ('X_umap_harmony',     'Harmony'),
    ('X_umap_scvi',        'scVI'),
]
 
for col, (key, title) in enumerate(methods):
    adata.obsm['X_umap'] = adata.obsm[key]
    sc.pl.umap(adata, color='sample_id',   ax=axes[0, col],
               show=False, title=f'{title} — by sample',
               legend_loc='right margin', size=3)
    sc.pl.umap(adata, color='CD3E',        ax=axes[1, col],
               show=False, title=f'{title} — CD3E expression',
               cmap='viridis', size=3)
 
plt.tight_layout()
plt.savefig('results/figures/integration_comparison.png', dpi=300)
plt.close()

D:\Projects\tumor-immune-scrna-atlas


In [3]:

 
# In the notebook:
from scib_metrics.benchmark import Benchmarker
 
bm = Benchmarker(
    adata,
    batch_key='sample_id',
    label_key='egfr_status',     # biology label - we want this preserved
    embedding_obsm_keys=['X_pca', 'X_pca_harmony', 'X_scVI'],
    n_jobs=4,
)
bm.benchmark()
df = bm.get_results(min_max_scale=False)
df.to_csv('results/tables/integration_metrics.csv')
print(df)

c:\Users\akksh\.conda\envs\scrna\Lib\site-packages\scib_metrics\benchmark\_core.py:193: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  sc.tl.pca(self._adata, svd_solver=self._solver, use_highly_variable=False)
Embeddings:   0%|          | 0/3 [00:00<?, ?it/s]c:\Users\akksh\.conda\envs\scrna\Lib\site-packages\scib_metrics\metrics\_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)
Embeddings:  33%|███▎      | 1/3 [01:33<03:06, 93.07s/it]c:\Users\akksh\.conda\envs\scrna\Lib\site-packages\scib_metrics\metrics\_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  

                Isolated labels        KMeans NMI        KMeans ARI  \
Embedding                                                             
X_pca                  0.509773           0.00758         -0.005175   
X_pca_harmony          0.511752           0.00175         -0.003236   
X_scVI                  0.49094          0.001456         -0.002642   
Metric Type    Bio conservation  Bio conservation  Bio conservation   

               Silhouette label             cLISI              BRAS  \
Embedding                                                             
X_pca                  0.514684          0.995801          0.734439   
X_pca_harmony          0.501808          0.367326          0.804699   
X_scVI                 0.502634          0.351408          0.940029   
Metric Type    Bio conservation  Bio conservation  Batch correction   

                          iLISI              KBET Graph connectivity  \
Embedding                                                              
X_

scVI wins every batch correction metric clearly. The most diagnostic one is Graph connectivity (0.914) — this measures whether cells of the same type actually connect across patients in the neighborhood graph. Harmony's 0.466 is poor: cells of the same type from different patients are still partially stranded from each other. This explains the dangling tail in the Harmony UMAP.






Use X_scVI for clustering. It has the best overall score, dramatically better graph connectivity (meaning your neighborhood graph will actually group T cells from all 10 patients together, not just T cells from the same patient), and bio conservation on par with Harmony.

# Why scVI over Harmony?
 scVI models count overdispersion with a negative binomial and learns a latent space directly comparable across batches. It also scales when adding new datasets. Harmony is faster but operates post-hoc on PCs of normalized data. For this project I used a pre-trained scVI model from scvi-hub fine-tuned with query-mode training — a common production approach when training from scratch isn't feasible. scIB metrics confirmed the choice: scVI scored higher on both batch correction and biological signal preservation

In [4]:
adata.obsm['X_umap'] = adata.obsm['X_umap_scvi']  # set scVI as default
adata.write('data/interim/05_integrated.h5ad')